# 📘 Unity Catalog Deep Dive
## Governance, Permissions, Lineage & Data Discovery

**What you'll learn:**
- Unity Catalog architecture (metastore, catalog, schema, table)
- Three-level namespace: catalog.schema.table
- Permissions model (GRANT/REVOKE, ownership)
- Row-level and column-level security
- Data lineage tracking
- Data discovery and tagging
- External locations and storage credentials
- Best practices for multi-team governance

**Prerequisites:** Notebooks 01 (SQL), 04 (Medallion)
**Study Order:** 12

---

> ⚠️ Unity Catalog features are limited on Community Edition.
> This notebook is primarily conceptual with runnable SQL where possible.

---

---
# 📗 Section 1: Unity Catalog Architecture

## The Three-Level Namespace

```
┌─────────────────────────────────────────────────────────────┐
│                    METASTORE                                  │
│  (Top-level container — one per Databricks account/region)   │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐     │
│  │   CATALOG    │  │   CATALOG    │  │   CATALOG    │     │
│  │  (prod)      │  │  (staging)   │  │  (dev)       │     │
│  ├──────────────┤  ├──────────────┤  ├──────────────┤     │
│  │ ┌──────────┐ │  │ ┌──────────┐ │  │ ┌──────────┐ │     │
│  │ │  SCHEMA  │ │  │ │  SCHEMA  │ │  │ │  SCHEMA  │ │     │
│  │ │ (sales)  │ │  │ │ (sales)  │ │  │ │ (sales)  │ │     │
│  │ ├──────────┤ │  │ ├──────────┤ │  │ ├──────────┤ │     │
│  │ │• orders  │ │  │ │• orders  │ │  │ │• orders  │ │     │
│  │ │• customers│ │  │ │• customers│ │  │ │• customers│ │     │
│  │ │• products│ │  │ │• products│ │  │ │• products│ │     │
│  │ └──────────┘ │  │ └──────────┘ │  │ └──────────┘ │     │
│  └──────────────┘  └──────────────┘  └──────────────┘     │
│                                                              │
└─────────────────────────────────────────────────────────────┘

Access pattern: SELECT * FROM prod.sales.orders
                             ^^^^^ ^^^^^ ^^^^^^
                            catalog schema table
```

## Key Concepts

| Concept | What It Is | Example |
|---------|-----------|---------|
| **Metastore** | Top-level container (one per region) | us-east-1 metastore |
| **Catalog** | Logical grouping (like a database server) | prod, staging, dev |
| **Schema** | Namespace within catalog (like a database) | sales, marketing, finance |
| **Table** | Data object | orders, customers |
| **View** | Virtual table (SQL query) | active_customers |
| **Volume** | File storage (replaces DBFS) | /Volumes/prod/raw/files/ |
| **Function** | UDF registered in catalog | mask_email() |

## Permissions Model

```sql
-- Grant SELECT on a table
GRANT SELECT ON TABLE prod.sales.orders TO `analyst-team`;

-- Grant all on a schema
GRANT ALL PRIVILEGES ON SCHEMA prod.sales TO `data-engineers`;

-- Grant usage on catalog (required to see it)
GRANT USAGE ON CATALOG prod TO `all-users`;

-- Row-level security
ALTER TABLE prod.sales.orders SET ROW FILTER filter_by_region ON (region);

-- Column masking
ALTER TABLE prod.sales.customers 
  ALTER COLUMN email SET MASK mask_email;
```

In [0]:
# Unity Catalog permission model simulation
class UnityCatalogSimulator:
    """Simulates Unity Catalog permission checks."""
    
    def __init__(self):
        self.permissions = {}  # (principal, object) → set of privileges
        self.objects = {}  # object_path → metadata
    
    def create_catalog(self, name, owner):
        self.objects[name] = {"type": "catalog", "owner": owner}
        self.permissions[(owner, name)] = {"ALL PRIVILEGES"}
    
    def create_schema(self, catalog, schema, owner):
        path = f"{catalog}.{schema}"
        self.objects[path] = {"type": "schema", "owner": owner}
        self.permissions[(owner, path)] = {"ALL PRIVILEGES"}
    
    def create_table(self, catalog, schema, table, owner):
        path = f"{catalog}.{schema}.{table}"
        self.objects[path] = {"type": "table", "owner": owner}
        self.permissions[(owner, path)] = {"ALL PRIVILEGES"}
    
    def grant(self, privilege, object_path, principal):
        key = (principal, object_path)
        if key not in self.permissions:
            self.permissions[key] = set()
        self.permissions[key].add(privilege)
    
    def check_access(self, principal, object_path, privilege):
        key = (principal, object_path)
        privs = self.permissions.get(key, set())
        return privilege in privs or "ALL PRIVILEGES" in privs
    
    def show_grants(self, principal):
        grants = []
        for (p, obj), privs in self.permissions.items():
            if p == principal:
                grants.append({"object": obj, "privileges": list(privs)})
        return grants


# Demo
uc = UnityCatalogSimulator()

# Setup
uc.create_catalog("prod", "admin")
uc.create_schema("prod", "sales", "admin")
uc.create_table("prod", "sales", "orders", "admin")
uc.create_table("prod", "sales", "customers", "admin")

# Grant permissions
uc.grant("USAGE", "prod", "analyst-team")
uc.grant("USAGE", "prod.sales", "analyst-team")
uc.grant("SELECT", "prod.sales.orders", "analyst-team")
uc.grant("SELECT", "prod.sales.customers", "analyst-team")

uc.grant("USAGE", "prod", "engineer-team")
uc.grant("ALL PRIVILEGES", "prod.sales", "engineer-team")

print("🔐 UNITY CATALOG PERMISSIONS")
print("=" * 60)

# Check access
checks = [
    ("analyst-team", "prod.sales.orders", "SELECT"),
    ("analyst-team", "prod.sales.orders", "MODIFY"),
    ("engineer-team", "prod.sales.orders", "MODIFY"),
    ("intern", "prod.sales.orders", "SELECT"),
]

for principal, obj, priv in checks:
    allowed = uc.check_access(principal, obj, priv)
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    print(f"   {status}: {principal} → {priv} on {obj}")

print("\n📋 Grants for analyst-team:")
for g in uc.show_grants("analyst-team"):
    print(f"   {g['object']}: {g['privileges']}")


---
# 📗 Section 2: Data Lineage & Discovery

## What is Data Lineage?

Lineage tracks WHERE data came from and WHERE it goes:

```
Source Systems          Bronze              Silver              Gold
┌──────────┐      ┌──────────┐      ┌──────────┐      ┌──────────┐
│ MySQL    │─────▶│raw_orders│─────▶│clean_    │─────▶│daily_    │
│ orders   │      │          │      │orders    │      │revenue   │
└──────────┘      └──────────┘      └──────────┘      └──────────┘
                                          │
                                          ▼
                                    ┌──────────┐
                                    │customer_ │
                                    │segments  │
                                    └──────────┘
```

Unity Catalog automatically tracks lineage for:
- Table-to-table dependencies (which tables feed which)
- Column-level lineage (which columns derive from which)
- Notebook/job lineage (which code produced which table)

## Data Discovery

Unity Catalog provides:
- **Search**: Find tables by name, description, or tags
- **Tags**: Label tables (e.g., "pii", "financial", "deprecated")
- **Comments**: Document tables and columns
- **Lineage graph**: Visual upstream/downstream dependencies

```sql
-- Add documentation
COMMENT ON TABLE prod.sales.orders IS 'All customer orders since 2020';
COMMENT ON COLUMN prod.sales.orders.customer_id IS 'FK to customers table';

-- Add tags
ALTER TABLE prod.sales.customers SET TAGS ('contains_pii', 'gdpr_relevant');
```

---
# 📗 Section 2: Unity Catalog Architecture

## The Three-Level Namespace

```
METASTORE (one per Databricks account/region)
  └── CATALOG: prod
        ├── SCHEMA: sales
        │     ├── TABLE: orders
        │     ├── TABLE: customers
        │     └── VIEW: active_customers
        ├── SCHEMA: finance
        │     └── TABLE: transactions
        └── SCHEMA: ml_features
              └── TABLE: customer_features

Access: SELECT * FROM prod.sales.orders
```

## Securable Objects

| Object | Privileges | Example |
|--------|-----------|---------|
| Metastore | CREATE CATALOG, USE CATALOG | Top-level admin |
| Catalog | USE CATALOG, CREATE SCHEMA | `GRANT USE CATALOG ON prod TO analysts` |
| Schema | USE SCHEMA, CREATE TABLE | `GRANT USE SCHEMA ON prod.sales TO analysts` |
| Table | SELECT, MODIFY, ALL PRIVILEGES | `GRANT SELECT ON TABLE prod.sales.orders TO analysts` |
| View | SELECT | `GRANT SELECT ON VIEW prod.sales.active_customers TO analysts` |
| Volume | READ_VOLUME, WRITE_VOLUME | `GRANT READ_VOLUME ON VOLUME prod.raw.files TO engineers` |
| Function | EXECUTE | `GRANT EXECUTE ON FUNCTION prod.utils.mask_email TO analysts` |

## Permission Inheritance

```
GRANT USAGE ON CATALOG prod TO `analyst-group`;
GRANT USAGE ON SCHEMA prod.sales TO `analyst-group`;
GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;

-- analyst-group can now: SELECT * FROM prod.sales.orders
-- analyst-group CANNOT: INSERT, UPDATE, DELETE, DROP
-- analyst-group CANNOT: access prod.finance (no USAGE on that schema)
```

In [0]:
%sql
-- Unity Catalog SQL examples
-- (These work on paid Databricks tiers with Unity Catalog enabled)

-- Show all catalogs
SHOW CATALOGS;

-- Show schemas in a catalog
SHOW SCHEMAS IN prod;

-- Show tables in a schema
SHOW TABLES IN prod.sales;

-- Grant permissions
-- GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;
-- GRANT USAGE ON CATALOG prod TO `analyst-group`;

-- Check your permissions
SHOW GRANTS ON TABLE techmart_dw.orders;

## Row-Level and Column-Level Security

### Row Filter (restrict which rows a user sees)

```sql
-- Create a filter function
CREATE FUNCTION prod.security.filter_by_region(region STRING)
RETURNS BOOLEAN
RETURN IF(IS_MEMBER('admin-group'), TRUE, region = current_user_region());

-- Apply to table
ALTER TABLE prod.sales.orders
SET ROW FILTER prod.security.filter_by_region ON (region);
```

### Column Masking (hide sensitive data)

```sql
-- Create masking function
CREATE FUNCTION prod.security.mask_email(email STRING)
RETURNS STRING
RETURN IF(IS_MEMBER('pii-access'), email, CONCAT(LEFT(email, 2), '***@***.com'));

-- Apply to column
ALTER TABLE prod.sales.customers
ALTER COLUMN email SET MASK prod.security.mask_email;
```

## Data Lineage

Unity Catalog automatically tracks:
- Which tables read from which tables
- Which notebooks/jobs created which tables
- Column-level lineage (which columns derive from which)

```sql
-- Query lineage (system tables)
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';
```

---
*Notebook 12 of the Databricks Data Engineering series*
*Study Order: 12*

In [0]:
# Unity Catalog concepts summary
print("Unity Catalog Quick Reference")
print("=" * 60)

uc_concepts = {
    "Namespace": "catalog.schema.table (3-level hierarchy)",
    "Metastore": "One per Databricks account/region -- top-level container",
    "Catalog": "Logical grouping (like a database server) -- prod, staging, dev",
    "Schema": "Namespace within catalog (like a database) -- sales, finance",
    "Table": "Data object -- orders, customers",
    "Volume": "File storage (replaces DBFS) -- /Volumes/prod/raw/files/",
    "External Location": "Cloud storage path registered in UC",
    "Storage Credential": "Cloud credentials for accessing external storage",
}

print("\nKey Concepts:")
for concept, description in uc_concepts.items():
    print(f"  {concept}: {description}")

print("\nPermission Hierarchy:")
print("  METASTORE > CATALOG > SCHEMA > TABLE/VIEW/VOLUME/FUNCTION")
print("  Must grant USAGE on each level before granting on objects within")

print("\nBest Practices:")
best_practices = [
    "Separate catalogs for dev/staging/prod",
    "Use groups (not individual users) for permissions",
    "Tag PII columns for compliance tracking",
    "Enable lineage for audit trails",
    "Use Unity Catalog Volumes instead of DBFS",
    "Set up external locations for cloud storage access",
]
for bp in best_practices:
    print(f"  * {bp}")


---
# 📗 Section 3: External Locations and Volumes

## External Locations

External locations register cloud storage paths in Unity Catalog:

```sql
-- Create storage credential
CREATE STORAGE CREDENTIAL my_s3_cred
WITH (AWS_IAM_ROLE = 'arn:aws:iam::123456789:role/databricks-role');

-- Create external location
CREATE EXTERNAL LOCATION my_data_lake
URL 's3://my-bucket/data/'
WITH (STORAGE CREDENTIAL my_s3_cred);

-- Grant access
GRANT READ FILES ON EXTERNAL LOCATION my_data_lake TO `data-engineers`;
```

## Unity Catalog Volumes

Volumes replace DBFS for file storage:

```python
# Old way (DBFS -- deprecated)
dbutils.fs.ls("dbfs:/FileStore/data/")

# New way (Unity Catalog Volumes)
dbutils.fs.ls("/Volumes/prod/raw/files/")

# Create a volume
spark.sql("CREATE VOLUME prod.raw.files")

# Write to volume
df.write.format("parquet").save("/Volumes/prod/raw/files/orders/")
```

## System Tables for Governance

```sql
-- Audit: who accessed what data
SELECT * FROM system.access.audit
WHERE action_name = 'SELECT'
  AND request_params.table_full_name = 'prod.sales.customers'
ORDER BY event_time DESC LIMIT 100;

-- Lineage: what tables depend on orders?
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';

-- Billing: compute usage by user
SELECT user_identity.email, SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1 ORDER BY 2 DESC;
```

In [0]:
%sql
-- Unity Catalog exploration queries
-- Show all catalogs you have access to
SHOW CATALOGS;

-- Show schemas in techmart_dw (our working catalog)
SHOW SCHEMAS IN techmart_dw;

-- Show tables
SHOW TABLES IN techmart_dw;

-- Describe a table (schema + properties)
DESCRIBE TABLE EXTENDED techmart_dw.orders;

In [0]:
# Unity Catalog best practices
print("Unity Catalog Best Practices")
print("=" * 60)

best_practices = {
    "Namespace": [
        "Use 3-level namespace: catalog.schema.table",
        "Separate catalogs for dev/staging/prod",
        "Use descriptive schema names (sales, finance, ml_features)",
    ],
    "Permissions": [
        "Use groups, not individual users",
        "Grant USAGE on catalog AND schema before table access",
        "Use least-privilege principle",
        "Review permissions quarterly",
    ],
    "PII Protection": [
        "Tag PII columns with system tags",
        "Apply column masking for sensitive data",
        "Use row filters for data residency",
        "Log all PII access in audit tables",
    ],
    "Storage": [
        "Use Unity Catalog Volumes instead of DBFS",
        "Register external locations for cloud storage",
        "Use managed tables when possible (UC manages lifecycle)",
    ],
}

for category, practices in best_practices.items():
    print(f"\n{category}:")
    for p in practices:
        print(f"  * {p}")


---
# Section 4: Row-Level and Column-Level Security

## Row-Level Security (RLS)

Restrict which rows a user can see based on their attributes.

```sql
-- Step 1: Create a filter function
CREATE FUNCTION prod.security.filter_by_region(region STRING)
RETURNS BOOLEAN
RETURN IF(IS_MEMBER('admin-group'), TRUE, region = current_user_region());

-- Step 2: Apply to table
ALTER TABLE prod.sales.orders
SET ROW FILTER prod.security.filter_by_region ON (region);

-- Result:
-- Admin sees ALL rows
-- User in US-East sees only US-East rows
-- User in EU-West sees only EU-West rows
```

## Column-Level Security (Data Masking)

```sql
-- Step 1: Create masking function
CREATE FUNCTION prod.security.mask_email(email STRING)
RETURNS STRING
RETURN IF(IS_MEMBER('pii-access-group'), email,
          CONCAT(LEFT(email, 2), '***@***.com'));

-- Step 2: Apply to column
ALTER TABLE prod.sales.customers
ALTER COLUMN email SET MASK prod.security.mask_email;

-- PII team sees:  alice@example.com
-- Analyst sees:   al***@***.com
```

## Common Masking Patterns

| Data Type | Masking | Example |
|-----------|---------|---------|
| Email | Show first 2 chars | al***@***.com |
| Phone | Show last 4 digits | ***-***-1234 |
| SSN | Show last 4 digits | ***-**-5678 |
| Credit card | Show last 4 digits | ****-****-****-1234 |
| Salary | Round to nearest $10K | $80,000 |

In [0]:
# Unity Catalog security simulation
import re

class DataMasker:
    def __init__(self, user_groups):
        self.groups = set(user_groups)

    def mask_email(self, email):
        if "pii-access-group" in self.groups:
            return email
        if email and "@" in email:
            local, domain = email.split("@", 1)
            return f"{local[:2]}***@***.com"
        return "***@***.com"

    def mask_phone(self, phone):
        if "pii-access-group" in self.groups:
            return phone
        digits = re.sub(r"\D", "", phone or "")
        return f"***-***-{digits[-4:]}" if len(digits) >= 4 else "***-***-****"

    def apply(self, record):
        masked = dict(record)
        if "email" in masked:
            masked["email"] = self.mask_email(masked["email"])
        if "phone" in masked:
            masked["phone"] = self.mask_phone(masked["phone"])
        return masked

customer = {
    "customer_id": 42,
    "name": "Alice Smith",
    "email": "alice.smith@example.com",
    "phone": "555-867-5309",
    "city": "New York",
}

print("Column-Level Security (Data Masking)")
print("=" * 60)
for role_name, groups in [
    ("PII Access Team", ["pii-access-group", "analyst-group"]),
    ("Regular Analyst", ["analyst-group"]),
    ("External Auditor", []),
]:
    masker = DataMasker(groups)
    masked = masker.apply(customer)
    print(f"\n{role_name}:")
    for key, value in masked.items():
        changed = " <-- masked" if str(value) != str(customer[key]) else ""
        print(f"  {key}: {value}{changed}")


---
# Section 5: Data Lineage and System Tables

## Data Lineage

Unity Catalog automatically tracks where data comes from and where it goes.

```sql
-- Find all tables that read from orders
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';

-- Find what feeds into daily_revenue
SELECT * FROM system.access.table_lineage
WHERE target_table_full_name = 'prod.gold.daily_revenue';

-- Audit: who accessed customer data
SELECT user_identity.email, action_name, event_time
FROM system.access.audit
WHERE request_params.table_full_name = 'prod.sales.customers'
ORDER BY event_time DESC
LIMIT 100;

-- Billing: compute usage by user
SELECT user_identity.email, SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1 ORDER BY 2 DESC;
```

## Unity Catalog Best Practices

| Practice | Why |
|----------|-----|
| Separate catalogs for dev/staging/prod | Isolate environments |
| Use groups, not individual users | Easier to manage at scale |
| Grant USAGE on catalog AND schema | Required before table access |
| Tag PII columns | Enables compliance tracking |
| Enable lineage | Automatic audit trail |
| Use Volumes instead of DBFS | DBFS is deprecated |

In [0]:
# Unity Catalog quick reference
print("Unity Catalog Quick Reference")
print("=" * 60)

uc_sql = {
    "Show all catalogs": "SHOW CATALOGS;",
    "Show schemas in catalog": "SHOW SCHEMAS IN prod;",
    "Show tables in schema": "SHOW TABLES IN prod.sales;",
    "Describe table": "DESCRIBE TABLE EXTENDED prod.sales.orders;",
    "Show table properties": "SHOW TBLPROPERTIES prod.sales.orders;",
    "Grant SELECT": "GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;",
    "Grant USAGE on catalog": "GRANT USAGE ON CATALOG prod TO `analyst-group`;",
    "Grant USAGE on schema": "GRANT USAGE ON SCHEMA prod.sales TO `analyst-group`;",
    "Revoke access": "REVOKE SELECT ON TABLE prod.sales.orders FROM `analyst-group`;",
    "Show grants": "SHOW GRANTS ON TABLE prod.sales.orders;",
    "Create volume": "CREATE VOLUME prod.raw.files;",
    "List volume files": "LIST '/Volumes/prod/raw/files/';",
}

for description, sql in uc_sql.items():
    print(f"  {description}:")
    print(f"    {sql}")
    print()


---
# Section 4: Row-Level and Column-Level Security

## Row-Level Security (RLS)

Restrict which rows a user can see based on their attributes.

```sql
-- Step 1: Create a filter function
CREATE FUNCTION prod.security.filter_by_region(region STRING)
RETURNS BOOLEAN
RETURN IF(IS_MEMBER('admin-group'), TRUE, region = current_user_region());

-- Step 2: Apply to table
ALTER TABLE prod.sales.orders
SET ROW FILTER prod.security.filter_by_region ON (region);

-- Result:
-- Admin sees ALL rows
-- User in US-East sees only US-East rows
-- User in EU-West sees only EU-West rows
```

## Column-Level Security (Data Masking)

```sql
-- Step 1: Create masking function
CREATE FUNCTION prod.security.mask_email(email STRING)
RETURNS STRING
RETURN IF(IS_MEMBER('pii-access-group'), email,
          CONCAT(LEFT(email, 2), '***@***.com'));

-- Step 2: Apply to column
ALTER TABLE prod.sales.customers
ALTER COLUMN email SET MASK prod.security.mask_email;

-- PII team sees:  alice@example.com
-- Analyst sees:   al***@***.com
```

## Common Masking Patterns

| Data Type | Masking | Example |
|-----------|---------|---------|
| Email | Show first 2 chars | al***@***.com |
| Phone | Show last 4 digits | ***-***-1234 |
| SSN | Show last 4 digits | ***-**-5678 |
| Credit card | Show last 4 digits | ****-****-****-1234 |
| Salary | Round to nearest $10K | $80,000 |

In [0]:
# Unity Catalog security simulation
import re

class DataMasker:
    def __init__(self, user_groups):
        self.groups = set(user_groups)

    def mask_email(self, email):
        if "pii-access-group" in self.groups:
            return email
        if email and "@" in email:
            local, domain = email.split("@", 1)
            return f"{local[:2]}***@***.com"
        return "***@***.com"

    def mask_phone(self, phone):
        if "pii-access-group" in self.groups:
            return phone
        digits = re.sub(r"\D", "", phone or "")
        return f"***-***-{digits[-4:]}" if len(digits) >= 4 else "***-***-****"

    def apply(self, record):
        masked = dict(record)
        if "email" in masked:
            masked["email"] = self.mask_email(masked["email"])
        if "phone" in masked:
            masked["phone"] = self.mask_phone(masked["phone"])
        return masked

customer = {
    "customer_id": 42,
    "name": "Alice Smith",
    "email": "alice.smith@example.com",
    "phone": "555-867-5309",
    "city": "New York",
}

print("Column-Level Security (Data Masking)")
print("=" * 60)
for role_name, groups in [
    ("PII Access Team", ["pii-access-group", "analyst-group"]),
    ("Regular Analyst", ["analyst-group"]),
    ("External Auditor", []),
]:
    masker = DataMasker(groups)
    masked = masker.apply(customer)
    print(f"\n{role_name}:")
    for key, value in masked.items():
        changed = " <-- masked" if str(value) != str(customer[key]) else ""
        print(f"  {key}: {value}{changed}")


---
# Section 5: Data Lineage and System Tables

## Data Lineage

Unity Catalog automatically tracks where data comes from and where it goes.

```sql
-- Find all tables that read from orders
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';

-- Find what feeds into daily_revenue
SELECT * FROM system.access.table_lineage
WHERE target_table_full_name = 'prod.gold.daily_revenue';

-- Audit: who accessed customer data
SELECT user_identity.email, action_name, event_time
FROM system.access.audit
WHERE request_params.table_full_name = 'prod.sales.customers'
ORDER BY event_time DESC
LIMIT 100;

-- Billing: compute usage by user
SELECT user_identity.email, SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1 ORDER BY 2 DESC;
```

## Unity Catalog Best Practices

| Practice | Why |
|----------|-----|
| Separate catalogs for dev/staging/prod | Isolate environments |
| Use groups, not individual users | Easier to manage at scale |
| Grant USAGE on catalog AND schema | Required before table access |
| Tag PII columns | Enables compliance tracking |
| Enable lineage | Automatic audit trail |
| Use Volumes instead of DBFS | DBFS is deprecated |

In [0]:
# Unity Catalog quick reference
print("Unity Catalog Quick Reference")
print("=" * 60)

uc_sql = {
    "Show all catalogs": "SHOW CATALOGS;",
    "Show schemas in catalog": "SHOW SCHEMAS IN prod;",
    "Show tables in schema": "SHOW TABLES IN prod.sales;",
    "Describe table": "DESCRIBE TABLE EXTENDED prod.sales.orders;",
    "Show table properties": "SHOW TBLPROPERTIES prod.sales.orders;",
    "Grant SELECT": "GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;",
    "Grant USAGE on catalog": "GRANT USAGE ON CATALOG prod TO `analyst-group`;",
    "Grant USAGE on schema": "GRANT USAGE ON SCHEMA prod.sales TO `analyst-group`;",
    "Revoke access": "REVOKE SELECT ON TABLE prod.sales.orders FROM `analyst-group`;",
    "Show grants": "SHOW GRANTS ON TABLE prod.sales.orders;",
    "Create volume": "CREATE VOLUME prod.raw.files;",
    "List volume files": "LIST '/Volumes/prod/raw/files/';",
}

for description, sql in uc_sql.items():
    print(f"  {description}:")
    print(f"    {sql}")
    print()


---
# Section 4: Row-Level and Column-Level Security

## Row-Level Security (RLS)

Restrict which rows a user can see based on their attributes.

```sql
-- Step 1: Create a filter function
CREATE FUNCTION prod.security.filter_by_region(region STRING)
RETURNS BOOLEAN
RETURN IF(IS_MEMBER('admin-group'), TRUE, region = current_user_region());

-- Step 2: Apply to table
ALTER TABLE prod.sales.orders
SET ROW FILTER prod.security.filter_by_region ON (region);

-- Result:
-- Admin sees ALL rows
-- User in US-East sees only US-East rows
-- User in EU-West sees only EU-West rows
```

## Column-Level Security (Data Masking)

```sql
-- Step 1: Create masking function
CREATE FUNCTION prod.security.mask_email(email STRING)
RETURNS STRING
RETURN IF(IS_MEMBER('pii-access-group'), email,
          CONCAT(LEFT(email, 2), '***@***.com'));

-- Step 2: Apply to column
ALTER TABLE prod.sales.customers
ALTER COLUMN email SET MASK prod.security.mask_email;

-- PII team sees:  alice@example.com
-- Analyst sees:   al***@***.com
```

## Common Masking Patterns

| Data Type | Masking | Example |
|-----------|---------|---------|
| Email | Show first 2 chars | al***@***.com |
| Phone | Show last 4 digits | ***-***-1234 |
| SSN | Show last 4 digits | ***-**-5678 |
| Credit card | Show last 4 digits | ****-****-****-1234 |
| Salary | Round to nearest $10K | $80,000 |

In [0]:
# Unity Catalog security simulation
import re

class DataMasker:
    def __init__(self, user_groups):
        self.groups = set(user_groups)

    def mask_email(self, email):
        if "pii-access-group" in self.groups:
            return email
        if email and "@" in email:
            local, domain = email.split("@", 1)
            return f"{local[:2]}***@***.com"
        return "***@***.com"

    def mask_phone(self, phone):
        if "pii-access-group" in self.groups:
            return phone
        digits = re.sub(r"\D", "", phone or "")
        return f"***-***-{digits[-4:]}" if len(digits) >= 4 else "***-***-****"

    def apply(self, record):
        masked = dict(record)
        if "email" in masked:
            masked["email"] = self.mask_email(masked["email"])
        if "phone" in masked:
            masked["phone"] = self.mask_phone(masked["phone"])
        return masked

customer = {
    "customer_id": 42,
    "name": "Alice Smith",
    "email": "alice.smith@example.com",
    "phone": "555-867-5309",
    "city": "New York",
}

print("Column-Level Security (Data Masking)")
print("=" * 60)
for role_name, groups in [
    ("PII Access Team", ["pii-access-group", "analyst-group"]),
    ("Regular Analyst", ["analyst-group"]),
    ("External Auditor", []),
]:
    masker = DataMasker(groups)
    masked = masker.apply(customer)
    print(f"\n{role_name}:")
    for key, value in masked.items():
        changed = " <-- masked" if str(value) != str(customer[key]) else ""
        print(f"  {key}: {value}{changed}")


---
# Section 5: Data Lineage and System Tables

## Data Lineage

Unity Catalog automatically tracks where data comes from and where it goes.

```sql
-- Find all tables that read from orders
SELECT * FROM system.access.table_lineage
WHERE source_table_full_name = 'prod.sales.orders';

-- Find what feeds into daily_revenue
SELECT * FROM system.access.table_lineage
WHERE target_table_full_name = 'prod.gold.daily_revenue';

-- Audit: who accessed customer data
SELECT user_identity.email, action_name, event_time
FROM system.access.audit
WHERE request_params.table_full_name = 'prod.sales.customers'
ORDER BY event_time DESC
LIMIT 100;

-- Billing: compute usage by user
SELECT user_identity.email, SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1 ORDER BY 2 DESC;
```

## Unity Catalog Best Practices

| Practice | Why |
|----------|-----|
| Separate catalogs for dev/staging/prod | Isolate environments |
| Use groups, not individual users | Easier to manage at scale |
| Grant USAGE on catalog AND schema | Required before table access |
| Tag PII columns | Enables compliance tracking |
| Enable lineage | Automatic audit trail |
| Use Volumes instead of DBFS | DBFS is deprecated |

In [0]:
# Unity Catalog quick reference
print("Unity Catalog Quick Reference")
print("=" * 60)

uc_sql = {
    "Show all catalogs": "SHOW CATALOGS;",
    "Show schemas in catalog": "SHOW SCHEMAS IN prod;",
    "Show tables in schema": "SHOW TABLES IN prod.sales;",
    "Describe table": "DESCRIBE TABLE EXTENDED prod.sales.orders;",
    "Show table properties": "SHOW TBLPROPERTIES prod.sales.orders;",
    "Grant SELECT": "GRANT SELECT ON TABLE prod.sales.orders TO `analyst-group`;",
    "Grant USAGE on catalog": "GRANT USAGE ON CATALOG prod TO `analyst-group`;",
    "Grant USAGE on schema": "GRANT USAGE ON SCHEMA prod.sales TO `analyst-group`;",
    "Revoke access": "REVOKE SELECT ON TABLE prod.sales.orders FROM `analyst-group`;",
    "Show grants": "SHOW GRANTS ON TABLE prod.sales.orders;",
    "Create volume": "CREATE VOLUME prod.raw.files;",
    "List volume files": "LIST '/Volumes/prod/raw/files/';",
}

for description, sql in uc_sql.items():
    print(f"  {description}:")
    print(f"    {sql}")
    print()


---
# 📗 Summary & Quick Reference

```
UNITY CATALOG HIERARCHY:
  Metastore → Catalog → Schema → Table/View/Volume/Function

PERMISSIONS:
  GRANT SELECT ON TABLE catalog.schema.table TO `group`;
  GRANT ALL PRIVILEGES ON SCHEMA catalog.schema TO `group`;
  GRANT USAGE ON CATALOG catalog TO `group`;  (required to see catalog)

SECURITY:
  Row-level: ALTER TABLE t SET ROW FILTER func ON (col);
  Column mask: ALTER TABLE t ALTER COLUMN c SET MASK func;

BEST PRACTICES:
  • Separate catalogs for dev/staging/prod
  • USAGE grants cascade (need USAGE on catalog to access schemas)
  • Use groups, not individual users
  • Tag PII columns for compliance
  • Enable lineage for audit trails
```

---
*Notebook 34 of the Databricks Data Engineering series*
*Study Order: 12*